# Lab 11 starter: forecasting food service revenue

This notebook is standalone. In the recap cell, the order and product tables are
loaded, the chapter's weekly company series is rebuilt, and the chapter's function
`forecast_all` is defined, so that you can apply the chapter's methods to one
product category.

The lab prompt is at the end of Chapter 11, under **Exercises**.

**Build lab.** Build the weekly revenue series for the food service category.
Decompose it and compare its six largest seasonal values with the company's. Then
compute the four forecasts for the last 13 weeks and report the scoreboard (MAE and
MAPE). State in two sentences the week and size of the largest seasonal value in
each series, as a percentage of the series' median week, and the method with the
lowest MAE and the lowest MAPE on this holdout.

**Evaluate lab.** Run the chapter's rolling backtest on the food service series, with
the same four holdouts. Report the MAE of each method in each quarter, and each
method's average MAE and MAPE over the four quarters. State in two sentences which
method has the lowest average MAE and which the lowest average MAPE, and whether ETS
meets the chapter's standard for this category: the lowest average MAE of all four
methods.

## Running this notebook

Run the setup cell below first, then the recap cell, then work down. In Google
Colab nothing needs to be installed beyond that cell. Locally, use the
`pyba-core` environment from Appendix A.

You will enter your work in the cells marked `# TODO`. Everything else is
provided. When you are unsure what a step produced, print it: `.head()` for a
table or a column, and the bare name for a number.

**If you are in Google Colab:** this notebook opened read-only from GitHub. Click
**Copy to Drive** in the toolbar (or *File > Save a copy in Drive*) before you
edit anything, then work in that copy. It saves to your Google Drive under
*Colab Notebooks*. Edits made to the read-only original are lost when the tab
closes.

In [ ]:
# Setup. Run this cell once per session. It installs this lab's packages;
# on Google Colab it also fetches the course data.
%pip install -q pandas statsmodels
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Recap: the Chapter 11 setup

The recap cell contains the chapter's code:

- `orders` is the order table, with one row per order line, and `products` is the
  product table, with one row per SKU and a `category` column.
- `weekly` is the chapter's weekly company revenue, 156 weeks.
- `HOLDOUT` is 13, the number of weeks in one holdout.
- `forecast_all(series, end)` is the chapter's function from the evaluation section.
  In it, the four methods are fitted on the weeks of `series` before position `end`.
  It returns two objects: the actual values of the 13 weeks from position `end` on,
  and a table with one column per method.

The six largest seasonal values of the company series are printed at the end, so
that you can compare the category with them in Step 5.

In [ ]:
# --- Chapter 11 recap: run this cell first ---
import pandas as pd

from pyba import DATA_DIR
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing

orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA_DIR / "pw_products.csv")

weekly = orders.set_index("order_date")["line_total"].resample("W").sum()
weekly.index.name = "week"

HOLDOUT = 13


def forecast_all(series, end):
    train = series.iloc[:end]                  # the weeks before the holdout
    test = series.iloc[end:end + HOLDOUT]      # the 13 holdout weeks
    f = pd.DataFrame(index=test.index)
    f["naive"] = train.iloc[-1]
    f["seasonal_naive"] = series.shift(52).iloc[end:end + HOLDOUT]
    f["moving_avg_8"] = train.iloc[-8:].mean()
    model = ExponentialSmoothing(train, trend="add", seasonal="add",
                                 seasonal_periods=52).fit()
    f["ETS"] = model.forecast(HOLDOUT)
    return test, f


company = seasonal_decompose(weekly, period=52)
print("company: the six largest seasonal values")
company.seasonal.iloc[:52].nlargest(6).round(0)

## Step 1: Add the category to the order lines

Make `lines`: `orders` merged with the `sku` and `category` columns of `products`,
as in Chapter 6. Use `on="sku"`, `how="left"`, and `validate="many_to_one"`, since
many order lines share one product. Then make `fs_lines`, the rows of `lines` whose
`category` is `"food service"`, with a boolean filter as in Chapter 3.

With a left join, every order line is kept. `lines` therefore has the same 140,371
rows as `orders` and one more column. `fs_lines` has fewer rows and the same
columns.

In [ ]:
# TODO: merge the category onto the order lines, then keep the food service lines
lines = None
fs_lines = None

# --- Look: the shapes, and the first food service lines ---
print(orders.shape, lines.shape, fs_lines.shape)
fs_lines[["order_id", "order_date", "sku", "category", "line_total"]].head(3)

## Step 2: Build the weekly food service series

Make `fs_weekly` from `fs_lines` exactly as `weekly` was made from `orders` in
the chapter: move `order_date` into the index, select `line_total`, resample to weeks
with `"W"`, and sum. Name the index `"week"`.

The result is a Series with one value per week. It should have 156 weeks, like the
company series, and its first value is the food service revenue of the week ending
July 9, 2023.

In [ ]:
# TODO: weekly food service revenue
fs_weekly = None

# --- Look: the number of weeks and the first three ---
print(len(fs_weekly), "weeks")
fs_weekly.head(3)

## Step 3: Decompose the series

Decompose `fs_weekly` with `seasonal_decompose` and `period=52`, as in the chapter.
Then print its six largest seasonal values, with `.iloc[:52]` and `nlargest(6)`, as
in the chapter for the company series.

To compare the size of the peaks, use the median week as a typical week for each
series, with `.median()` as in Chapter 4. For each series, print the median week and
the largest seasonal value as a percentage of it: the `.max()` of the seasonal
values, divided by the median, times 100. The company's seasonal values are
`company.seasonal`.

Because the seasonal values repeat every year, the first 52 weeks hold all of them.
The dates in the output are those of the first year.

In [ ]:
# TODO: decompose fs_weekly; print the medians, the two percentages, and its six largest seasonal values
fs_decomp = None

## Step 4: Score the four methods on the last 13 weeks

The last 13 weeks start at position `len(fs_weekly) - HOLDOUT`, which is 143. Call
`forecast_all` with that position to get `fs_test` (the actual revenue of the 13
weeks) and `fs_forecasts` (the four forecasts), as in the chapter's backtest
loop.

Then build the scoreboard with the chapter's loop over the columns of the
forecasts table: for each method, the MAE and the MAPE, one row per method, sorted
by MAE.

In [ ]:
# TODO: the four forecasts for the last 13 weeks, then the scoreboard
fs_test, fs_forecasts = None, None

## Step 5: Seasonal pattern and holdout result

Two sentences, each with the numbers from your output. First, for each series,
state the week of the largest seasonal value and that value as a percentage of the
median week (Step 3), and the number of the six largest seasonal values that fall
in January. Then name the method with the lowest MAE and the method with the lowest
MAPE on this holdout (Step 4), with their values.

---

**Seasonal pattern:** *replace with your answer*

**Lowest error on the last 13 weeks:** *replace with your answer*

---

## Step 6 (Evaluate lab): Run the rolling backtest

Use the chapter's backtest loop on `fs_weekly`. For each holdout start in
`[104, 117, 130, 143]`, call `forecast_all`, and for each method append one row
with the start date of the holdout, the method, the MAE, and the MAPE. Make the
DataFrame `fs_backtest` from the rows, and show its MAE values as a grid with
`.pivot_table`, one row per holdout and one column per method.

`fs_backtest` should have 16 rows: four holdouts times four methods.

In [ ]:
# TODO: rolling backtest on the food service series
fs_backtest = None

## Step 7: Average each method over the four quarters

Average the MAE and the MAPE of each method over the four holdouts with `.groupby`,
as in the chapter, and sort by MAE.

In [ ]:
# TODO: average MAE and MAPE per method

## Step 8: Comparison with the chapter's standard

Two sentences, each with the numbers from Step 7. First, name the method with the
lowest average MAE and the method with the lowest average MAPE. Then state whether
ETS meets the chapter's standard, the lowest average MAE of all four methods, and
give its difference from the baseline with the next lowest average MAE.

---

**Lowest average errors:** *replace with your answer*

**The chapter's standard:** *replace with your answer*

---

## Submission

Run **Restart and run all** and confirm the notebook executes top to bottom with no
errors. Then download it as `.ipynb` and upload it to the Lab 11 dropbox.

If you used an AI assistant, add a one-line note naming the tool and what it
generated, per the course policy in Appendix D.